# StreamVault profiling (stage 5)

Explores the films as TMDb gave them. It **calls the pipeline's own functions** and re-implements nothing.
Findings cite `outputs/quality_report.csv`, `quality_flags.csv` and `quality_skew.csv`, so every number
reproduces without Jupyter by running `python run_pipeline.py`.
Reads the cache only: no API calls.

In [9]:
import sys
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == "profiling" else Path.cwd()
sys.path.insert(0, str(ROOT))          # so config and pipeline import from the project root

import pandas as pd
import config
from pipeline.load_ids import load_and_classify_ids
from pipeline.flatten import build_tables
from pipeline.quality import quality_report

pd.set_option("display.width", 200)

In [10]:
# Load from cache (no API calls)
fetchable, quarantine = load_and_classify_ids(config.ID_LIST_PATH)
films, cast, crew, _ = build_tables(config.RAW_DIR)
manifest = pd.read_csv(config.OUTPUT_DIR / "fetch_manifest.csv")
len(films), len(cast), len(crew)

(112, 7827, 24932)

In [11]:
# The stage 5 checks
report, flags, skew = quality_report(films, manifest, quarantine)
report

,check,group,window,count,denominator,note
0,money_both_present,in_collection,all_years,96,96,
1,money_budget_missing_only,in_collection,all_years,0,96,
2,money_revenue_missing_only,in_collection,all_years,0,96,
3,money_both_missing,in_collection,all_years,0,96,
4,money_both_present,no_collection,all_years,16,16,
5,money_budget_missing_only,no_collection,all_years,0,16,
6,money_revenue_missing_only,no_collection,all_years,0,16,
7,money_both_missing,no_collection,all_years,0,16,
8,money_both_present,in_collection,from_2000,63,63,
9,money_budget_missing_only,in_collection,from_2000,0,63,


In [12]:
# Skew: mean vs median
skew

,group,window,n,mean,median,max,top_film_share,top_film_id
0,in_collection,all_years,96,6.545567e+08,578937111.0,2.942650e+09,0.046830,299534
1,no_collection,all_years,16,2.702712e+08,161802536.5,8.390306e+08,0.194025,27205
2,in_collection,from_2000,63,7.929864e+08,786636033.0,2.942650e+09,0.058902,299534
3,no_collection,from_2000,8,3.538082e+08,283195047.5,8.390306e+08,0.296428,27205


In [13]:
# Money by group
films.assign(group=films["collection_id"].notna().map({True: "in_collection", False: "no_collection"})) \
     .groupby("group")[["budget", "revenue"]].describe().T

group             in_collection     no_collection
budget  count              96.0              16.0
        mean   117231770.833333        55731250.0
        std     82958396.396149   60613603.739865
        min           6000000.0         1200000.0
        25%          53000000.0        16500000.0
        50%         101000000.0        25000000.0
        75%         152500000.0        69750000.0
        max         379000000.0       180000000.0
revenue count              96.0              16.0
        mean   654556698.145833    270271171.3125
        std    434483120.831549  278758937.243749
        min          41722424.0         7390108.0
        25%         370709953.0       45319272.25
        50%         578937111.0       161802536.5
        75%        927851235.25      371352140.25
        max        2942649778.0       839030630.0

In [14]:
# Films per decade
(films["release_year"] // 10 * 10).value_counts().sort_index()

release_year
1960     1
1970     5
1980    13
1990    22
2000    48
2010    23
Name: count, dtype: Int64

In [15]:
# Films per collection
films.groupby("collection_name")["movie_id"].count().sort_values(ascending=False)

collection_name
Harry Potter Collection                8
Star Wars Collection                   6
Pirates of the Caribbean Collection    5
Mission: Impossible Collection         5
Die Hard Collection                    4
Shrek Collection                       4
Indiana Jones Collection               4
Alien Collection                       4
The Avengers Collection                4
Toy Story Collection                   4
The Terminator Collection              4
Jurassic Park Collection               3
Ocean's Collection                     3
Ice Age Collection                     3
Iron Man Collection                    3
Back to the Future Collection          3
The Matrix Collection                  3
The Godfather Collection               3
The Hobbit Collection                  3
The Lord of the Rings Collection       3
The Bourne Collection                  3
The Dark Knight Collection             3
Kill Bill Collection                   2
Blade Runner Collection                1


In [16]:
# The 16 standalones (null-collection check by eye)
films.loc[films["collection_id"].isna(), ["movie_id", "title", "release_year", "revenue"]] \
     .sort_values("release_year")

,movie_id,title,release_year,revenue
65,429,"The Good, the Bad and the Ugly",1966,38900000
6,111,Scarface,1983,66023329
91,769,GoodFellas,1990,47072327
63,424,Schindler's List,1993,321365567
49,278,The Shawshank Redemption,1994,28341469
15,13,Forrest Gump,1994,677387716
87,680,Pulp Fiction,1994,213928762
69,550,Fight Club,1999,100853753
80,641,Requiem for a Dream,2000,7390108
92,77,Memento,2000,40060108
